In [28]:
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

# Student Inputs
LAST_NAME = "Asis"
STUDENT_ID = "TUPM-26-1217"
SEED_DIGIT = int(STUDENT_ID[-1]) 
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  

# Data Generation Rules
RECORDS_TO_PROCESS = 1000 + ID_SUM  
TARGET_CLASS_DIGIT = (SEED_DIGIT % 4) + 2  

# Paths
DATASET_PATH = Path("data/weblog.csv")
FILTERED_CSV_PATH = Path("output/filtered_5xx_logs.csv")
SUMMARY_JSON_PATH = Path("output/log_summary.json")
BACKUP_JSON_PATH = Path("backup/log_summary_backup.json")

# Data containers
records_processed = 0
target_records = []
status_counts = Counter()
url_counts = Counter()

# Read CSV line-by-line without loading entire file into memory
with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as file:
    reader = csv.DictReader(file)

    for row in reader:
        if records_processed >= RECORDS_TO_PROCESS:
            break

        records_processed += 1

        try:
            ip = row.get("IP", "").strip()
            timestamp = row.get("Time", "").strip()
            url = row.get("URL", "").strip()
            status_str = row.get("Staus", "").strip()

            status_code = int(status_str)

            # Check if status code belongs to target 2xx class (200-299)
            status_counts[status_code] += 1
            url_counts[url] += 1

            if (TARGET_CLASS_DIGIT * 100) <= status_code < ((TARGET_CLASS_DIGIT + 1) * 100):
                target_records.append({"IP": ip, "Time": timestamp, "URL": url, "Status": status_code})

        except (ValueError, TypeError):
            continue

# Export target records to filtered CSV file
with open(FILTERED_CSV_PATH, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["IP", "Time", "URL", "Status"])
    writer.writeheader()
    writer.writerows(target_records)

# Identify top 5 most requested URLs
top_5_urls = [url for url, _ in url_counts.most_common(5)]

# Write summary to JSON file
summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": records_processed,
    "target_status_class": f"{TARGET_CLASS_DIGIT}xx",
    "records_in_target_class": len(target_records),
    "count_per_status_code": dict(status_counts),
    "top_5_requested_urls": top_5_urls,
}

with open(SUMMARY_JSON_PATH, mode="w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

# Create backup of JSON file and verify existence
shutil.copy(SUMMARY_JSON_PATH, BACKUP_JSON_PATH)
backup_verified = BACKUP_JSON_PATH.exists()

print("=" * 70)
print(f"{'EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS REPORT':^70}")
print("=" * 70)
print(f"Records Processed      : {records_processed}")
print(f"Target Status Class    : {TARGET_CLASS_DIGIT}xx")
print(f"Records in Target Class: {len(target_records)}")
print(f"Count per Status Code  : {dict(status_counts)}")
print(f"Top 5 Requested URLs   : {top_5_urls}")
print(f"Filtered CSV File      : {FILTERED_CSV_PATH.name}")
print(f"JSON Summary File      : {SUMMARY_JSON_PATH.name}")
print(f"Backup Verification    : {'VERIFIED' if backup_verified else 'FAILED'}")
print("=" * 70)

          EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS REPORT           
Records Processed      : 1019
Target Status Class    : 5xx
Records in Target Class: 0
Count per Status Code  : {200: 842, 302: 143, 304: 21, 206: 8, 404: 5}
Top 5 Requested URLs   : ['GET /login.php HTTP/1.1', 'GET /home.php HTTP/1.1', 'GET / HTTP/1.1', 'GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1', 'GET /css/normalize.css HTTP/1.1']
Filtered CSV File      : filtered_5xx_logs.csv
JSON Summary File      : log_summary.json
Backup Verification    : VERIFIED
